In [37]:
import sys
import argparse
import os
sys.path.append(os.path.expanduser("~/websites/mapedia"))
from modules import DBHandler, DBUpdater
import pandas as pd
import numpy as np
from modules.db_handler.DBConfig import INTER_CITY_LEARNING_SOURCE, INTRA_CITY_LEARNING_SOURCE
INTRA_CITY_LEARNING_PRESET_CONFIDENCE = 0.8

In [2]:
db_handler = DBHandler()
db_handler.connect_to_db()
db_updater = DBUpdater(db_handler)

In [4]:
metadata_path  = os.path.join('./data/imputed_data/', f"jakarta_imputedBy_jakarta.parquet")
metadata = pd.read_parquet(metadata_path)
metadata.head()

,idx,source,target,pgr_id,osm_id,oneway,road_type,nlanes,width,length,...,imputed_avg_speed_weekend_08-12,avg_speed_weekend_12-16,pred_avg_speed_weekend_12-16,imputed_avg_speed_weekend_12-16,avg_speed_weekend_16-20,pred_avg_speed_weekend_16-20,imputed_avg_speed_weekend_16-20,avg_speed_weekend_20-24,pred_avg_speed_weekend_20-24,imputed_avg_speed_weekend_20-24
0,0,52860,7874,98,4705045,1.0,2.0,4.0,10.0,56.769815,...,8.897243,8.955077,41.542290,8.955077,11.846818,48.506863,11.846818,5.271908,47.002674,5.271908
1,1,30445,87959,212,4705043,1.0,2.0,1.0,4.0,293.579384,...,7.279488,7.610105,22.613220,7.610105,11.480000,24.226820,11.480000,9.215501,24.076851,9.215501
2,2,1359,102290,213,4705046,1.0,2.0,2.0,6.0,12.779379,...,8.752268,9.169477,28.441622,9.169477,12.175380,31.166859,12.175380,15.253763,30.977135,15.253763
3,3,47817,58347,38783,4705046,1.0,2.0,2.0,6.0,76.590609,...,8.752268,9.169477,42.077873,9.169477,12.175380,45.876884,12.175380,15.253763,48.486752,15.253763
4,4,7874,92444,47381,4705045,1.0,2.0,4.0,10.0,28.160970,...,8.897243,8.955077,34.823906,8.955077,11.846818,38.916023,11.846818,5.271908,38.942097,5.271908


In [5]:
metadata.columns

Index(['idx', 'source', 'target', 'pgr_id', 'osm_id', 'oneway', 'road_type',
       'nlanes', 'width', 'length', 'geometry', 'max_speed', 'min_speed',
       'nlanes_cls', 'highway_id', 'pred_road_type', 'pred_nlanes_cls',
       'pred_oneway', 'pred_width', 'pred_max_speed', 'pred_min_speed',
       'imputed_road_type', 'imputed_nlanes_cls', 'imputed_oneway',
       'imputed_width', 'imputed_max_speed', 'imputed_min_speed',
       'avg_speed_weekday_00-04', 'pred_avg_speed_weekday_00-04',
       'imputed_avg_speed_weekday_00-04', 'avg_speed_weekday_04-08',
       'pred_avg_speed_weekday_04-08', 'imputed_avg_speed_weekday_04-08',
       'avg_speed_weekday_08-12', 'pred_avg_speed_weekday_08-12',
       'imputed_avg_speed_weekday_08-12', 'avg_speed_weekday_12-16',
       'pred_avg_speed_weekday_12-16', 'imputed_avg_speed_weekday_12-16',
       'avg_speed_weekday_16-20', 'pred_avg_speed_weekday_16-20',
       'imputed_avg_speed_weekday_16-20', 'avg_speed_weekday_20-24',
       'pred_avg_s

In [9]:
pred_metadata = metadata[['osm_id',
        'pred_road_type', 'pred_nlanes_cls',
       'pred_oneway', 'pred_width', 'pred_max_speed', 'pred_min_speed']].copy()
pred_metadata.rename(columns={
    'pred_road_type': 'road_type', 
    'pred_nlanes_cls': 'nlanes',
    'pred_oneway': 'oneway', 
    'pred_width': 'width', 
    'pred_max_speed': 'max_speed', 
    'pred_min_speed': 'min_speed'
}, inplace=True)

In [19]:
for c in ['road_type', 'nlanes', 'oneway', 'width', 'max_speed', 'min_speed']:
    pred_metadata[f'{c}_source'] = INTRA_CITY_LEARNING_SOURCE
    pred_metadata[f'{c}_conf'] = INTRA_CITY_LEARNING_PRESET_CONFIDENCE

In [20]:
speed_metadata = metadata[['osm_id',
       'pred_avg_speed_weekday_00-04',
       'pred_avg_speed_weekday_04-08',
       'pred_avg_speed_weekday_08-12',
       'pred_avg_speed_weekday_12-16',
       'pred_avg_speed_weekday_16-20',
       'pred_avg_speed_weekday_20-24',
       'pred_avg_speed_weekend_00-04',
       'pred_avg_speed_weekend_04-08',
       'pred_avg_speed_weekend_08-12',
       'pred_avg_speed_weekend_12-16',
       'pred_avg_speed_weekend_16-20',
       'pred_avg_speed_weekend_20-24']].copy()



In [21]:
# Period label → (start_hour, end_hour)
PERIOD_HOURS = {
    "00-04": range(0, 4),
    "04-08": range(4, 8),
    "08-12": range(8, 12),
    "12-16": range(12, 16),
    "16-20": range(16, 20),
    "20-24": range(20, 24),
}

# weekday=0 means Monday in pandas; 0-4 = weekday, 5-6 = weekend
WEEKDAY_DAYS = list(range(5))   # 0-4
WEEKEND_DAYS = list(range(5, 7)) # 5-6

# 1. One period-block: 6 periods × their hour counts = 24 values
#    Each period value repeated for its hours
period_cols_weekday = [f"pred_avg_speed_weekday_{p}" for p in PERIOD_HOURS]
period_cols_weekend = [f"pred_avg_speed_weekend_{p}" for p in PERIOD_HOURS]
period_lengths      = [len(h) for h in PERIOD_HOURS.values()]  # [4,4,4,4,4,4]

def build_day_vector(row, cols):
    """24 values: each period value repeated for its hour count"""
    return np.repeat([row[c] for c in cols], period_lengths)  # (24,)

def build_week_vector(row):
    """168 values: 5× weekday-day + 2× weekend-day"""
    day = build_day_vector(row, period_cols_weekday)  # (24,)
    end = build_day_vector(row, period_cols_weekend)  # (24,)
    return np.concatenate([np.tile(day, 5), np.tile(end, 2)])  # (168,)

def build_speed_array(row):
    """672 values: week vector repeated 4× for seasons"""
    return np.tile(build_week_vector(row), 4)  # (672,)

# Apply once per road
speed_matrix = np.stack(speed_metadata.apply(build_speed_array, axis=1))  # (N, 672)

In [22]:
pred_metadata["avg_speed"] = list(speed_matrix)
pred_metadata["avg_speed_source"] = [np.full(672, INTRA_CITY_LEARNING_SOURCE, dtype=object)] * len(pred_metadata)
pred_metadata["avg_speed_conf"]   = list(np.full((len(pred_metadata), 672), INTRA_CITY_LEARNING_PRESET_CONFIDENCE, dtype=np.float32))
dynamic_pred_metadata = pred_metadata[['osm_id', 'avg_speed', 'avg_speed_source', 'avg_speed_conf']].copy()
static_pred_metadata = pred_metadata.drop(columns=['avg_speed', 'avg_speed_source', 'avg_speed_conf'])

In [23]:
dynamic_pred_metadata = dynamic_pred_metadata.set_index("osm_id").rename_axis(None)
static_pred_metadata = static_pred_metadata.set_index("osm_id").rename_axis(None)

In [25]:
static_pred_metadata.osm_id.unique()

AttributeError: 'DataFrame' object has no attribute 'osm_id'

In [27]:
static_pred_metadata.loc[221344726]

,road_type,nlanes,oneway,width,max_speed,min_speed,road_type_source,road_type_conf,nlanes_source,nlanes_conf,oneway_source,oneway_conf,width_source,width_conf,max_speed_source,max_speed_conf,min_speed_source,min_speed_conf
221344726,1,0,0.0,8.689913,628.269104,3.547565,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8
221344726,1,0,0.0,8.708470,628.192444,3.566581,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8
221344726,1,0,0.0,8.781711,629.321167,3.595962,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8
221344726,1,0,0.0,8.687202,628.573059,3.548753,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8
221344726,1,0,0.0,8.961991,632.583740,3.640257,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8,4,0.8


In [28]:
dynamic_pred_metadata.loc[221344726]

,avg_speed,avg_speed_source,avg_speed_conf
221344726,"[26.188695907592773, 26.188695907592773, 26.18...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
221344726,"[26.300649642944336, 26.300649642944336, 26.30...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
221344726,"[26.599430084228516, 26.599430084228516, 26.59...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
221344726,"[26.203197479248047, 26.203197479248047, 26.20...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."
221344726,"[27.210365295410156, 27.210365295410156, 27.21...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, ..."


In [31]:
dynamic_pred_metadata.shape

(512991, 3)

In [34]:
osm_ids = list(dynamic_pred_metadata.index.unique())

In [35]:
road_attributes = db_updater.db_handler.roads_from_ids(osm_ids)

In [36]:
road_attributes.head()

,osm_id,oneway,road_type,width,nlanes,max_speed,min_speed,avg_speed,oneway_source,road_type_source,...,min_speed_source,avg_speed_source,oneway_conf,road_type_conf,width_conf,nlanes_conf,max_speed_conf,min_speed_conf,avg_speed_conf,geometry
4705040,11893428,NaN,1.0,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...",NaN,0.0,...,None,"[None, None, None, None, None, None, None, Non...",NaN,0.95,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...","LINESTRING (-79.807 42, -79.807 42.001, -79.80..."
4705043,11893437,NaN,0.0,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...",NaN,0.0,...,None,"[None, None, None, None, None, None, None, Non...",NaN,0.95,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...","LINESTRING (-77.722 40.142, -77.722 40.143, -7..."
4705045,11893446,NaN,1.0,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...",NaN,0.0,...,None,"[None, None, None, None, None, None, None, Non...",NaN,0.95,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...","LINESTRING (-80.108 42.102, -80.108 42.102, -8..."
4705046,11893451,NaN,1.0,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...",NaN,0.0,...,None,"[None, None, None, None, None, None, None, Non...",NaN,0.95,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...","LINESTRING (-77.658 40.25, -77.658 40.25, -77...."
8151584,16647200,NaN,0.0,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...",NaN,0.0,...,None,"[None, None, None, None, None, None, None, Non...",NaN,0.95,NaN,NaN,NaN,None,"[None, None, None, None, None, None, None, Non...","LINESTRING (-83.304 35.006, -83.304 35.006, -8..."


In [24]:
db_updater.update_database_new(static_attr=static_pred_metadata, temporal_attr=dynamic_pred_metadata)

Updating static road attributes:  15%|█▍        | 75569/512991 [00:20<01:56, 3758.81it/s]


KeyError: 221344726

In [38]:
import geopandas as gpd
edges_path  = os.path.join('./data/raw_data', f"jakarta_edges.parquet")
edges = gpd.read_parquet(edges_path)

In [39]:
edges

,source,target,pgr_id,osm_id,oneway,road_type,nlanes,width,length,geometry,max_speed,min_speed
0,52860,7874,98,4705045,1.0,2.0,4.0,10.000000,56.769815,"LINESTRING (106.84 -6.1673, 106.84 -6.1677)",NaN,NaN
1,30445,87959,212,4705043,1.0,2.0,1.0,4.000000,293.579384,"LINESTRING (106.84 -6.1678, 106.84 -6.1679, 10...",NaN,NaN
2,1359,102290,213,4705046,1.0,2.0,2.0,6.000000,12.779379,"LINESTRING (106.84 -6.1681, 106.84 -6.1682)",NaN,NaN
3,47817,58347,38783,4705046,1.0,2.0,2.0,6.000000,76.590609,"LINESTRING (106.84 -6.1669, 106.84 -6.167, 106...",NaN,NaN
4,7874,92444,47381,4705045,1.0,2.0,4.0,10.000000,28.160970,"LINESTRING (106.84 -6.1677, 106.84 -6.168)",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
512986,554572,3126680,4265252,1323780385,1.0,3.0,1.0,7.670237,5.210311,"LINESTRING (106.84 -6.1815, 106.84 -6.1815)",NaN,NaN
512987,562231,3126704,4265281,1318113065,0.0,1.0,1.0,2.319501,28.838543,"LINESTRING (106.76 -6.1815, 106.76 -6.1814, 10...",NaN,NaN
512988,1705639,3126735,4265318,1317986765,0.0,0.0,0.0,3.734968,27.817565,"LINESTRING (106.76 -6.1698, 106.76 -6.1697)",NaN,NaN
512989,1590678,3126756,4265341,1317609752,0.0,0.0,1.0,3.889288,30.114436,"LINESTRING (106.69 -6.1917, 106.69 -6.1917, 10...",NaN,NaN
